# 🌋 KALI: Sovereign Weight-Bake (Phase 52)
This notebook performs the actual fine-tuning of KALI's LLM using the **Unsloth** framework for maximum efficiency on T4/L4 GPUs. 

**Sovereignty Goals:**
1. Replace external API dependencies with local weights.
2. Fine-tune on verified `knowledge_atoms.jsonl` data.
3. Achieve 90%+ alignment with the Great Council's reasoning.

In [ ]:
# 1. Install KALI Sovereign Dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets jsonlines

In [ ]:
# 2. Clone KALI Repository & Data
!git clone https://github.com/Adityavanjre/Project-K.git
%cd Project-K
!mkdir -p data

# Upload your training_data.jsonl to the 'data/' folder if not using samples
import os
if not os.path.exists('data/training_data.jsonl'):
    print("WARNING: training_data.jsonl NOT FOUND. Ensure you upload it to Project-K/data/ before proceeding.")

In [ ]:
# 3. Logic: Prepare Dataset for KALI Prompt Matrix
import json
from datasets import Dataset

def format_kami_interaction(examples):
    inputs = examples["user"]
    outputs = examples["assistant"]
    texts = []
    for i, o in zip(inputs, outputs):
        text = f"Below is an interaction between a user and KALI, a sovereign AI mentor.\n\n### User:\n{i}\n\n### KALI:\n{o}"
        texts.append(text)
    return { "text" : texts, }

with open("data/training_data.jsonl", "r") as f:
    raw = [json.loads(l) for l in f if l.strip()]

formatted = []
for entry in raw:
    msgs = entry.get("messages", [])
    u = next((m["content"] for m in msgs if m["role"] == "user"), "")
    a = next((m["content"] for m in msgs if m["role"] == "assistant"), "")
    if u and a: formatted.append({"user": u, "assistant": a})

dataset = Dataset.from_list(formatted).map(format_kami_interaction, batched=True)
print(f"[*] KALI Matrix Loaded: {len(dataset)} verified atoms.")

In [ ]:
# 4. Initialize Sovereign Model (Llama-3-8B-Instruct)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = True,
)

In [ ]:
# 5. Execute Training (Weight-Bake)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Adjust based on dataset size
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        output_dir = "kali_lora_outputs",
    ),
)

trainer.train()

In [ ]:
# 6. Export Sovereign Weights
model.save_pretrained_lora("kali_weights")
tokenizer.save_pretrained("kali_weights")
print("[*] KALI Weights Baked & Saved to PROJECT-K/kali_weights.")